# **Clase 12 - 🎯 Repaso integrador de cara al Primer Parcial**

### Laboratorio de Métodos Cuantitativos aplicados a la Gestión
**Tecnicatura Universitaria en Gestión y Análisis de Datos — FCE UBA**
Viernes 25 de septiembre de 2026 · Clase virtual · Unidades 1, 2 y 3

## ¿Qué vamos a hacer en esta clase?

Esta clase **no trae temas nuevos**. Todo lo que está acá ya lo vimos entre la clase 00 y la 11.

Lo que hacemos hoy es otra cosa: **ordenar** lo que ya saben, ponerlo en cuadros que se puedan
mirar de un saque, y practicar la parte que el parcial realmente evalúa, que es
**interpretar resultados**, no escribir código.

La idea es que al terminar la clase tengan esta notebook como material de estudio para los días
que quedan hasta el martes.

## Objetivos de la clase

Al terminar, deberían poder:

1. **Leer** un array, un `DataFrame`, una derivada y una salida de optimización, y decir qué dicen
   sobre la organización del caso.
2. **Distinguir** las herramientas que se parecen y se confunden: `*` contra `@`, `.loc` contra
   `.iloc`, `linprog` contra `PuLP`.
3. **Resolver a mano** el circuito completo de optimización: derivar, igualar a cero, verificar con
   la segunda derivada e interpretar el resultado en pesos y unidades.
4. **Detectar errores** en un bloque de código o en un filtro: qué hace, qué devuelve y dónde está
   la falla.

## 🗺️ Qué entra en el parcial (y cómo se corrige)

El parcial es el **martes 29 de septiembre a las 17:00**, dura 3 horas, es **en papel** y
**no se usa computadora**. Entran las **Unidades 1, 2 y 3**.

| Bloque de la materia | Clases | Qué se pide en el parcial |
|---|---|---|
| Python, datos y archivos | 00, 01 | Leer un bloque de código y decir qué devuelve |
| Visualización | 02 | **Interpretar** un gráfico ya hecho |
| Funciones económicas y equilibrio | 03, 04 | Plantear costo, ingreso, beneficio; leer un equilibrio |
| Vectores y matrices | 05 | Operar e interpretar `axis`, `*` contra `@` |
| Filtrado y muestreo | 06 | Decir si un filtro está bien hecho y qué deja afuera |
| Sistemas de ecuaciones | 07 | Plantear el sistema e interpretar la solución |
| Programación lineal | 08 | Plantear el modelo y **leer la salida** |
| Derivadas y variaciones | 09 | Derivar, interpretar el signo |
| Marginales y elasticidades | 10 | Costo, ingreso y beneficio marginal; elasticidad |
| Optimización | 11 | Máximos y mínimos con criterio de segunda derivada |

> ⚠️ **El examen no evalúa escribir programas.** Evalúa la lectura económica y de gestión de los
> resultados, el criterio con que se construye y se depura una base, y el procedimiento matemático.
> Sobre código se pide **operar con elementos de código sin escribirlo**: qué hace un bloque, qué
> devuelve, dónde está el error y cómo se corrige.

> 📌 **Toda respuesta numérica sin justificación vale cero.** Se corrige el procedimiento y la
> lectura del resultado, no la cifra final sola.

## 📖 Cómo usar esta notebook

- **Durante la clase:** seguimos los bloques en orden. Cada bloque tiene una parte de teoría
  condensada (los cuadros) y un ejemplo corto que corremos juntos.
- **Después de la clase:** los cuadros funcionan como machete conceptual para repasar, y al final
  hay **ejercicios tipo parcial** con la resolución escondida (hacelos antes de abrirla).
- **Ojo:** el machete es para estudiar **ahora**, no para el examen. El martes no se puede llevar
  ningún material.

Arranquemos importando lo que vamos a usar en toda la clase.

In [ ]:
# Paso 1: importamos las librerías de siempre
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
from scipy.optimize import linprog

# PuLP no viene preinstalado en todos los entornos.
# Si esta celda falla en Colab, descomentá la línea del pip install y volvé a correrla.
# !pip install pulp -q
try:
    import pulp
    PULP_OK = True
except ImportError:
    PULP_OK = False
    print("PuLP no está instalado: la celda de PuLP se va a saltear.")

print("Librerías listas. NumPy", np.__version__, "| pandas", pd.__version__)

---
# 🔢 Bloque 1 — NumPy: arrays y vectores

## La teoría en cuatro líneas

Un **array** de NumPy es una fila (o una tabla) de números **todos del mismo tipo**. Eso es lo que
lo hace rápido y lo que lo diferencia de una lista de Python.

| Concepto | Qué significa | Cómo se ve |
|---|---|---|
| **Vector** | Un array de una dimensión: una lista de números | `np.array([10, 20, 30])` |
| **Matriz** | Un array de dos dimensiones: filas y columnas | `np.array([[1, 2], [3, 4]])` |
| **`dtype`** | El tipo de dato del array; es **uno solo** para todo el array | `int64`, `float64` |
| **`shape`** | La forma: `(filas, columnas)` | `(3,)` es un vector de 3; `(2, 2)` una matriz 2×2 |
| **Vectorización** | Operar sobre todo el array de una, sin `for` | `ventas * 1.21` |

La **vectorización** es la razón de ser de NumPy: en lugar de recorrer con un `for`, se le pide la
operación al array completo y NumPy la aplica elemento por elemento.

In [ ]:
# Un vector: las ventas de tres sucursales, en miles de pesos
ventas = np.array([120, 340, 275])

print("El vector:      ", ventas)
print("Su forma:       ", ventas.shape, "-> un vector de 3 elementos")
print("Su tipo de dato:", ventas.dtype)

# Vectorización: le sumamos el IVA a TODAS las sucursales de una sola vez, sin for
ventas_con_iva = ventas * 1.21
print("\nCon IVA:        ", ventas_con_iva.round(2))

# Sumar dos vectores: se suma posición por posición (elemento a elemento)
costos = np.array([80, 200, 190])
print("Margen bruto:   ", ventas - costos)

## ✖️ `*` contra `@`: la confusión más común del bloque

Son **dos multiplicaciones distintas** y dan resultados distintos:

| Operador | Cómo se llama | Qué hace | Cuándo se usa |
|---|---|---|---|
| `*` | Producto **elemento a elemento** | Multiplica cada posición con la que está en la misma posición | Precios × cantidades de cada producto |
| `@` | Producto **matricial** (o escalar entre vectores) | Multiplica y **suma**: devuelve un total | Facturación total, insumo-producto |

La regla práctica: **si el resultado tiene que ser un total, va `@`; si tiene que seguir siendo una
lista de valores, va `*`.**

In [ ]:
precios    = np.array([100, 250, 80])   # precio de cada producto
cantidades = np.array([10,   4,  25])   # unidades vendidas de cada producto

# Con * : cuánto facturó CADA producto (sigue habiendo tres números)
por_producto = precios * cantidades
print("Facturación por producto:", por_producto)

# Con @ : cuánto facturó el negocio EN TOTAL (multiplica y suma: un solo número)
total = precios @ cantidades
print("Facturación total:       ", total)

# Es exactamente lo mismo que sumar el resultado de *
print("Comprobación:            ", por_producto.sum())

## 🧭 `axis`: la pregunta que confunde a todo el mundo

Cuando la tabla tiene dos dimensiones hay que decirle a NumPy **en qué dirección** resumir:

| Qué escribo | En qué dirección se mueve | Qué pregunta responde | Qué devuelve |
|---|---|---|---|
| `M.sum()` | Todo | ¿Cuál es el total general? | **Un** número |
| `M.sum(axis=0)` | Hacia abajo, recorre **filas** | ¿Cuánto por **columna**? | Un valor por columna |
| `M.sum(axis=1)` | Hacia el costado, recorre **columnas** | ¿Cuánto por **fila**? | Un valor por fila |

> 💡 **Truco para no dudar:** `axis=0` es el eje de las filas, así que **las filas desaparecen** y
> queda un resultado por columna. `axis=1` es el eje de las columnas, así que **las columnas
> desaparecen** y queda un resultado por fila. El eje que nombrás es el que se consume.

In [ ]:
# Filas = sucursales (Centro, Norte, Sur) | Columnas = trimestres (T1, T2, T3, T4)
M = np.array([[120, 135, 150, 160],
              [340, 310, 360, 380],
              [275, 290, 285, 300]])

print("Forma de la tabla:", M.shape, "-> 3 sucursales x 4 trimestres\n")
print("Total general:              ", M.sum())
print("Total por trimestre (axis=0):", M.sum(axis=0), "<- 4 valores, uno por columna")
print("Total por sucursal (axis=1): ", M.sum(axis=1), "<- 3 valores, uno por fila")

# Y el promedio por sucursal, con el mismo criterio
print("\nPromedio por sucursal:      ", M.mean(axis=1).round(1))

> ⚠️ **Errores clásicos de este bloque (los que se ven en el parcial)**
>
> | Error | Qué pasa | Cómo se detecta |
> |---|---|---|
> | Usar `*` donde iba `@` | Devuelve una lista en lugar de un total | El resultado tiene más de un número cuando se pedía uno solo |
> | Confundir `axis=0` con `axis=1` | El resultado tiene la cantidad de valores equivocada | Contá: ¿hay un valor por sucursal o por trimestre? |
> | Sumar arrays de distinto largo | Error de forma, o un resultado inesperado por *broadcasting* | Mirá siempre `.shape` antes de operar |
> | Suponer el `dtype` | Si el array es `int`, una división puede truncar | Revisá `.dtype` cuando el número no cierra |

---
# 🐼 Bloque 2 — pandas: indexar y filtrar

## Las dos piezas de pandas

| Pieza | Qué es | Analogía |
|---|---|---|
| **`Series`** | Una columna con su índice | Una columna de Excel |
| **`DataFrame`** | Una tabla: varias `Series` que comparten índice | Una hoja de Excel entera |

Y la trampa de notación que más cuesta:

| Cómo escribo | Qué me devuelve |
|---|---|
| `df["ventas"]` | Corchete **simple**: una `Series` (una columna sola) |
| `df[["ventas"]]` | Corchete **doble**: un `DataFrame` de una columna |
| `df[["ventas", "costos"]]` | Corchete doble: un `DataFrame` de dos columnas |

> 💡 **Por qué importa en el parcial:** si la consigna pide "una tabla" y el código devuelve una
> `Series`, eso es parte de la respuesta. El doble corchete es lo que mantiene la forma de tabla.

In [ ]:
# Armamos una tabla chica para trabajar en todo el bloque
datos = pd.DataFrame({
    "sucursal":  ["Centro", "Norte", "Sur", "Oeste", "Centro", "Norte"],
    "mes":       ["ene", "ene", "ene", "ene", "feb", "feb"],
    "ventas":    [120, 340, 275, 90, 135, 310],
    "empleados": [4, 9, 7, 3, 4, 9],
})

print("Forma:", datos.shape, "-> 6 filas, 4 columnas\n")
datos

In [ ]:
# Corchete simple contra corchete doble: MISMA columna, distinta forma
una_serie = datos["ventas"]
una_tabla = datos[["ventas"]]

print("Con corchete simple ->", type(una_serie).__name__, "| forma:", una_serie.shape)
print("Con corchete doble  ->", type(una_tabla).__name__, "| forma:", una_tabla.shape)

## 🎯 `.loc` contra `.iloc`: la diferencia que hay que tener clarísima

| | `.loc` | `.iloc` |
|---|---|---|
| **Busca por** | **Etiqueta** (el nombre del índice o de la columna) | **Posición** (el número de orden, empezando en 0) |
| **Se lee** | "*location*": dónde está, por nombre | "*integer location*": dónde está, por número |
| **Ejemplo** | `datos.loc[0, "ventas"]` | `datos.iloc[0, 2]` |
| **El final del rango** | **Incluido**: `.loc[0:2]` trae 3 filas | **Excluido**: `.iloc[0:2]` trae 2 filas |
| **Acepta condiciones** | Sí: `datos.loc[datos["ventas"] > 200]` | No |

> ⚠️ **La trampa del rango** es el error que más cae: `.loc[0:2]` devuelve **tres** filas (la 0, la 1
> y la 2) porque incluye el final, mientras que `.iloc[0:2]` devuelve **dos** (la 0 y la 1), como
> cualquier rebanado de Python.

In [ ]:
# Mismo dato, dos caminos
print("Con .loc  (etiqueta 0, columna 'ventas'):", datos.loc[0, "ventas"])
print("Con .iloc (posición 0, posición 2):      ", datos.iloc[0, 2])

# La trampa del rango
print("\n.loc[0:2]  ->", len(datos.loc[0:2]), "filas (el final ESTÁ incluido)")
print(".iloc[0:2] ->", len(datos.iloc[0:2]), "filas (el final NO está incluido)")

## ↔️ ¿Está agarrando una fila o una columna?

Esta es una pregunta típica de parcial: te muestran un bloque y hay que decir qué devuelve.

| Cómo escribo | Qué agarra | Qué devuelve |
|---|---|---|
| `datos["ventas"]` | Una **columna** completa | `Series` de 6 valores |
| `datos.loc[2]` | Una **fila** completa | `Series` con los 4 campos de esa fila |
| `datos.loc[2, "ventas"]` | Una **celda** | Un solo valor |
| `datos.loc[:, "ventas"]` | Todas las filas de una columna | `Series` (los dos puntos significan "todas") |
| `datos.loc[2, :]` | Todas las columnas de una fila | `Series` |

> 💡 **Cómo darse cuenta:** en `.loc[fila, columna]` el **primer** lugar siempre es la fila y el
> **segundo** la columna. Si hay un solo argumento, es la fila.

In [ ]:
print("Una columna -> datos['ventas']:")
print(datos["ventas"].to_string(), "\n")

print("Una fila -> datos.loc[2]:")
print(datos.loc[2].to_string(), "\n")

print("Una celda -> datos.loc[2, 'ventas']:", datos.loc[2, "ventas"])

## ✂️ Filtrar: ¿está bien filtrado?

Filtrar es quedarse con las filas que cumplen una condición. Se hace con una **máscara booleana**:
una columna de `True` y `False` que dice, fila por fila, si entra o no.

El proceso tiene dos pasos, y conviene verlos separados:

1. **La condición** `datos["ventas"] > 200` devuelve la máscara.
2. **El filtro** `datos[datos["ventas"] > 200]` usa la máscara para quedarse con las filas `True`.

In [ ]:
# Paso 1: la máscara (qué filas cumplen)
mascara = datos["ventas"] > 200
print("La máscara:")
print(mascara.to_string(), "\n")

# Paso 2: el filtro (quedarse con esas filas)
grandes = datos[mascara]
print("Filas que quedaron:", len(grandes), "de", len(datos))
grandes

### ⚠️ Los cuatro errores de filtrado que hay que saber reconocer

| Error | Cómo se escribe mal | Cómo se escribe bien | Qué pasa si está mal |
|---|---|---|---|
| **`and` en lugar de `&`** | `df[(a > 1) and (b < 2)]` | `df[(a > 1) & (b < 2)]` | Tira error: `and` no sabe trabajar con columnas enteras |
| **Faltan los paréntesis** | `df[a > 1 & b < 2]` | `df[(a > 1) & (b < 2)]` | El `&` se evalúa antes que el `>` y el resultado es otro |
| **`=` en lugar de `==`** | `df[df["mes"] = "ene"]` | `df[df["mes"] == "ene"]` | Error de sintaxis: uno asigna, el otro compara |
| **Filtrar y no mirar cuánto quedó** | — | `len(df_filtrado)` | Te quedás con 0 filas (o con todas) y no te enterás |

> 📌 **Lo que se pregunta en el parcial no es "escribí el filtro", es "¿este filtro está bien?".**
> Para responder eso hay que decir tres cosas: qué condición aplica, **cuántas filas deja** y
> **cuáles deja afuera**.

In [ ]:
# Filtro con DOS condiciones, bien escrito: & entre paréntesis
correcto = datos[(datos["ventas"] > 200) & (datos["mes"] == "ene")]
print("Filtro correcto -> quedan", len(correcto), "filas:")
print(correcto[["sucursal", "mes", "ventas"]].to_string(index=False))

# El mismo filtro con | (o): cambia por completo lo que queda
con_o = datos[(datos["ventas"] > 200) | (datos["mes"] == "ene")]
print("\nCon | en lugar de & -> quedan", len(con_o), "filas. No es lo mismo.")

## 📊 `groupby`: resumir por categoría

`groupby` hace siempre lo mismo, en tres pasos: **parte** la tabla por una columna, **aplica** una
cuenta a cada grupo y **junta** los resultados.

Se lee así: `datos.groupby("sucursal")["ventas"].sum()` → *"agrupá por sucursal, mirá la columna
ventas y sumala"*.

In [ ]:
# Total vendido por sucursal
por_sucursal = datos.groupby("sucursal")["ventas"].sum()
print("Ventas por sucursal:")
print(por_sucursal.to_string(), "\n")

# Y un resumen con dos cuentas a la vez
resumen = datos.groupby("sucursal").agg(
    ventas_totales=("ventas", "sum"),
    meses_con_datos=("ventas", "count"),
    empleados=("empleados", "max"),
)
resumen

> 💡 **Cómo se interpreta un `groupby` en el parcial:** el índice del resultado son las **categorías**
> y los valores son la **cuenta pedida**. Antes de interpretar, verificá que la suma de los grupos dé
> el total general: si no da, el filtro o el agrupamiento dejaron filas afuera.

---
# 💰 Bloque 3 — Funciones económicas y lectura de gráficos

## El cuadro de definiciones

Todo el bloque económico de la materia se apoya en estas funciones. Conviene tenerlas de memoria:

| Función | Notación | Fórmula | Qué significa |
|---|---|---|---|
| Costo fijo | $CF$ | constante | Lo que se paga aunque no se produzca nada |
| Costo variable | $CV(q)$ | depende de $q$ | Lo que cuesta producir, crece con la cantidad |
| **Costo total** | $CT(q)$ | $CF + CV(q)$ | Todo lo que cuesta producir $q$ |
| **Costo medio** | $CMe(q)$ | $\dfrac{CT(q)}{q}$ | Cuánto cuesta **cada** unidad en promedio |
| **Costo marginal** | $CMg(q)$ | $\dfrac{dCT}{dq}$ | Cuánto cuesta producir **una unidad más** |
| **Ingreso total** | $IT(q)$ | $p \cdot q$ | Lo que entra por vender $q$ unidades |
| **Ingreso marginal** | $IMg(q)$ | $\dfrac{dIT}{dq}$ | Cuánto entra por vender **una unidad más** |
| **Beneficio** | $B(q)$ | $IT(q) - CT(q)$ | Lo que queda |

> ⚠️ **Costo medio y costo marginal no son lo mismo**, y es un error clásico mezclarlos. El medio
> **divide** (reparte el total entre las unidades); el marginal **deriva** (mira el cambio por una
> unidad extra).

In [ ]:
# Una fábrica: costo fijo de $5.000 y un costo variable que crece más que proporcionalmente
q = np.linspace(1, 100, 300)      # cantidades posibles

CF = 5000
def costo_total(q):  return CF + 40 * q + 0.5 * q**2
def ingreso(q):      return 200 * q - 0.5 * q**2     # el precio baja cuando se vende más
def beneficio(q):    return ingreso(q) - costo_total(q)

# Graficamos las tres juntas
plt.figure(figsize=(9, 5))
plt.plot(q, costo_total(q), label="Costo total CT(q)")
plt.plot(q, ingreso(q),     label="Ingreso total IT(q)")
plt.plot(q, beneficio(q),   label="Beneficio B(q)", linewidth=2)
plt.axhline(0, color="gray", linewidth=0.8)
plt.title("Costo, ingreso y beneficio")
plt.xlabel("Cantidad producida (unidades)")
plt.ylabel("Pesos")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 🧠 Lo que hay que ver en ese gráfico

Un gráfico así aparece en el parcial y se pide interpretarlo. Lo que se mira, en este orden:

1. **Dónde arranca cada curva.** El costo total no arranca en cero: arranca en el **costo fijo**
   ($5.000). El ingreso sí arranca en cero, porque sin vender no entra nada.
2. **Dónde se cruzan $IT$ y $CT$.** Ahí el beneficio es cero: son los **puntos de nivelación**. A la
   izquierda del primero se pierde plata, en el medio se gana.
3. **Dónde está el techo del beneficio.** Es el punto más alto de la curva de beneficio, y
   **no** coincide con el máximo del ingreso: lo que se maximiza es la diferencia, no la venta.
4. **Qué pasa al final.** Si el beneficio empieza a bajar, producir más **empeora** el resultado. Es
   el argumento económico de por qué existe una cantidad óptima.

> 📌 **En el parcial, describir la forma no alcanza.** "La curva sube y después baja" no es una
> interpretación. La interpretación es: *"conviene producir hasta ~q unidades, porque a partir de ahí
> cada unidad extra cuesta más de lo que suma al ingreso"*.

---
# 📈 Bloque 4 — Derivadas: primera, segunda, máximos y mínimos

## Qué dice una derivada, en castellano

La derivada $f'(x)$ mide **cuánto cambia $f$ cuando $x$ aumenta en una unidad**. En gestión eso es
directamente la lectura marginal: "si produzco una unidad más, ¿cuánto me cambia el costo?".

| Lo que veo | Qué significa | Cómo se lee en la organización |
|---|---|---|
| $f'(x) > 0$ | La función **crece** | Conviene seguir avanzando en esa dirección |
| $f'(x) < 0$ | La función **decrece** | Cada paso adicional empeora el resultado |
| $f'(x) = 0$ | **Punto crítico**: ni sube ni baja | Candidato a máximo o mínimo |
| $f''(x) < 0$ | Cóncava (forma de loma) | El punto crítico es un **MÁXIMO** |
| $f''(x) > 0$ | Convexa (forma de valle) | El punto crítico es un **MÍNIMO** |

> 💡 **Truco para no confundir la segunda derivada:** la loma ($\cap$) tiene el máximo arriba y su
> segunda derivada es **negativa**; el valle ($\cup$) tiene el mínimo abajo y su segunda derivada es
> **positiva**. Signo negativo → máximo. Signo positivo → mínimo.

## 🧮 La receta de optimización (esto es lo que hay que escribir en el parcial)

Siempre los mismos cuatro pasos. Conviene escribirlos numerados en la hoja:

1. **Plantear** la función a optimizar, con sus unidades.
2. **Derivar** e igualar a cero: $f'(q) = 0$. Resolver. Eso da el o los **puntos críticos** (la
   *condición de primer orden*).
3. **Verificar** con la segunda derivada: $f''(q) < 0$ es máximo, $f''(q) > 0$ es mínimo (la
   *condición de segundo orden*). **Este paso no se puede saltear**: sin él no está demostrado que
   sea un máximo.
4. **Interpretar**: reemplazar el óptimo en la función original y decir qué significa el número en
   pesos, unidades y decisión de gestión.

In [ ]:
# Mismo caso de la fábrica, ahora en simbólico con SymPy
q = sp.symbols("q", positive=True)

CT = 5000 + 40*q + 0.5*q**2      # costo total
IT = 200*q - 0.5*q**2            # ingreso total
B  = IT - CT                     # beneficio

print("Beneficio B(q) =", sp.simplify(B))

# PASO 2: primera derivada igualada a cero (condición de primer orden)
B1 = sp.diff(B, q)
print("\nB'(q)  =", sp.simplify(B1))

criticos = sp.solve(sp.Eq(B1, 0), q)
print("Puntos críticos:", criticos)

# PASO 3: segunda derivada (condición de segundo orden)
B2 = sp.diff(B, q, 2)
print("\nB''(q) =", B2, "-> es negativa, así que el punto crítico es un MÁXIMO")

# PASO 4: interpretar
q_opt = criticos[0]
print("\nCantidad óptima:  ", round(float(q_opt), 2), "unidades")
print("Beneficio máximo: $", round(float(B.subs(q, q_opt)), 2))

### 🔍 Las tres funciones de SymPy que hay que saber nombrar

| Función | Qué hace | Ejemplo |
|---|---|---|
| `sp.symbols("q")` | Crea la variable simbólica (sin esto no se puede derivar) | `q = sp.symbols("q")` |
| `sp.diff(f, q)` | Deriva `f` respecto de `q` | `sp.diff(B, q)` |
| `sp.diff(f, q, 2)` | Deriva **dos veces**: la segunda derivada | `sp.diff(B, q, 2)` |
| `sp.solve(ecuacion, q)` | Resuelve la ecuación | `sp.solve(sp.Eq(B1, 0), q)` |
| `f.subs(q, valor)` | Reemplaza la variable por un número | `B.subs(q, 80)` |
| `sp.lambdify(q, f)` | Convierte la expresión simbólica en una función que acepta arrays | para graficar |

> ⚠️ **El error que más se repite:** derivar y quedarse ahí. `sp.diff` devuelve la **función**
> derivada, no el óptimo. Para el óptimo hay que igualar a cero y resolver (`sp.solve`), y después
> **verificar con la segunda derivada**.

In [ ]:
# Visualizamos el óptimo: convertimos la expresión simbólica a numérica con lambdify
B_num = sp.lambdify(q, B, "numpy")
qs = np.linspace(1, 160, 400)

plt.figure(figsize=(9, 5))
plt.plot(qs, B_num(qs), label="Beneficio B(q)")
plt.axvline(float(q_opt), color="red", linestyle="--",
            label=f"q óptimo = {float(q_opt):.1f}")
plt.axhline(0, color="gray", linewidth=0.8)
plt.scatter([float(q_opt)], [float(B.subs(q, q_opt))], color="red", zorder=5, s=60)
plt.title("El beneficio y su máximo")
plt.xlabel("Cantidad producida (unidades)")
plt.ylabel("Beneficio (pesos)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
# ⚖️ Bloque 5 — Marginales y elasticidades

## Marginal = derivada del total

| Marginal | Se obtiene derivando | Qué responde |
|---|---|---|
| **Costo marginal** $CMg$ | $CT'(q)$ | ¿Cuánto **cuesta** la unidad siguiente? |
| **Ingreso marginal** $IMg$ | $IT'(q)$ | ¿Cuánto **entra** por la unidad siguiente? |
| **Beneficio marginal** $BMg$ | $B'(q) = IMg - CMg$ | ¿Cuánto **suma al resultado** la unidad siguiente? |

## 🎯 La regla de oro: $IMg = CMg$

El beneficio es máximo cuando el **ingreso marginal se iguala al costo marginal**. Y es lógico:

- Si $IMg > CMg$, la unidad siguiente **deja plata** → conviene producirla.
- Si $IMg < CMg$, la unidad siguiente **cuesta más de lo que trae** → conviene no producirla.
- El punto donde se cruzan es donde ya no se puede mejorar: es el **óptimo**.

> 💡 Esto es **exactamente lo mismo** que igualar $B'(q) = 0$, porque $B' = IMg - CMg$. Son dos
> caminos al mismo número, y en el parcial cualquiera de los dos sirve **si está justificado**.

In [ ]:
# Los marginales del mismo caso
CMg = sp.diff(CT, q)
IMg = sp.diff(IT, q)

print("CMg(q) =", CMg)
print("IMg(q) =", IMg)

# El óptimo por la regla IMg = CMg
q_regla = sp.solve(sp.Eq(IMg, CMg), q)[0]
print("\nDonde IMg = CMg ->  q =", round(float(q_regla), 2))
print("Por B'(q) = 0    ->  q =", round(float(q_opt), 2), " <- el mismo número")

# Lectura de una unidad más, en el óptimo
print("\nEn el óptimo: CMg = $", round(float(CMg.subs(q, q_regla)), 2),
      " | IMg = $", round(float(IMg.subs(q, q_regla)), 2))

## 📐 Elasticidad precio de la demanda

La elasticidad mide **cuánto reacciona la cantidad demandada cuando cambia el precio**, en términos
porcentuales:

$$\epsilon_p = \frac{\%\ \text{cambio en la cantidad}}{\%\ \text{cambio en el precio}}$$

| Valor (en módulo) | Cómo se llama | Qué significa para la gestión |
|---|---|---|
| $|\epsilon_p| > 1$ | **Elástica** | La cantidad reacciona **mucho**: si subís el precio, el ingreso **baja** |
| $|\epsilon_p| = 1$ | **Unitaria** | El ingreso está en su máximo: no cambia con el precio |
| $|\epsilon_p| < 1$ | **Inelástica** | La cantidad reacciona **poco**: si subís el precio, el ingreso **sube** |

> 📌 **La pregunta de parcial** no es calcular la elasticidad, es decidir con ella: *"¿conviene subir
> el precio?"*. Si la demanda es inelástica, sí (el ingreso sube). Si es elástica, no.

---
# 📊 Bloque 6 — Programación lineal: `linprog` y `PuLP`

## Anatomía de un problema de PL

Todo problema de programación lineal se plantea con las mismas cuatro piezas. **Escribirlas
explícitamente es lo que se corrige** en el parcial:

| Pieza | Qué es | Ejemplo |
|---|---|---|
| **Variables de decisión** | Lo que la organización elige, **con unidades** | $x$ = kilos de pan por día |
| **Función objetivo** | Lo que se maximiza o minimiza | $\max\ 120x + 200y$ |
| **Restricciones** | Los límites de recursos | $x + 2y \le 160$ (horas de horno) |
| **No negatividad** | No se produce en negativo | $x \ge 0,\ y \ge 0$ |

Vamos con un caso concreto para los dos solvers:

> **Caso.** Una panadería hace **pan** ($x$) y **facturas** ($y$), en kilos por día. El pan deja
> \$120 por kilo y las facturas \$200. Cada kilo de pan usa 1 hora de horno y 3 kg de masa; cada kilo
> de facturas, 2 horas de horno y 2 kg de masa. Hay **160 horas** de horno y **300 kg** de masa por
> día. ¿Cuánto conviene producir de cada cosa?

In [ ]:
# ---------- Con linprog (SciPy) ----------
# OJO: linprog SIEMPRE minimiza. Para MAXIMIZAR hay que poner los coeficientes en NEGATIVO.
c = [-120, -200]                    # maximizar 120x + 200y  ->  minimizar -120x - 200y

A_ub = [[1, 2],                     # horno:  1x + 2y <= 160
        [3, 2]]                     # masa:   3x + 2y <= 300
b_ub = [160, 300]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, None), (0, None)], method="highs")

print(res)

## 🔍 Cómo se lee la salida de `linprog`

Esta lectura es pregunta directa de parcial. Campo por campo:

| Campo | Qué dice | En este caso |
|---|---|---|
| `message` / `status` | Si encontró la solución. `status: 0` es óptimo | Terminó bien |
| `success` | `True` si hay solución óptima | `True` |
| **`fun`** | El valor de la función objetivo **minimizada** | Viene **negativo**: la ganancia real es su valor absoluto |
| **`x`** | Los valores óptimos de las variables, **en el orden en que las declaré** | `[70, 45]` → 70 kg de pan, 45 de facturas |
| `nit` | Iteraciones que hizo el algoritmo | Dato técnico, no se interpreta |
| `marginals` | Cuánto mejora el objetivo si se afloja una restricción (precio sombra) | Fuera del alcance del parcial |

> ⚠️ **La trampa del signo.** Como pusimos `c` en negativo para maximizar, `fun` sale negativo. La
> ganancia es $|fun|$. Si en el parcial aparece un `fun` negativo en un problema de maximización,
> **eso no es un error**: es la consecuencia de que `linprog` minimiza siempre.

> 📌 **El orden de `x` es el orden en que declaraste las variables.** `x: [70, 45]` no dice "pan" ni
> "facturas": lo dice tu planteo. Por eso el paso de definir las variables **con unidades** es parte
> de la respuesta.

In [ ]:
# Interpretación explícita de la salida de arriba
pan, facturas = res.x
print(f"Producir {pan:.0f} kg de pan y {facturas:.0f} kg de facturas por día.")
print(f"Ganancia máxima: ${-res.fun:,.0f} por día   <- ojo el signo: -fun")

# ¿Los recursos se usan del todo?
horno_usado = 1*pan + 2*facturas
masa_usada  = 3*pan + 2*facturas
print(f"\nHorno: {horno_usado:.0f} de 160 horas usadas")
print(f"Masa:  {masa_usada:.0f} de 300 kg usados")
print("\nLas dos restricciones se usan completas: son restricciones ACTIVAS.")

In [ ]:
# ---------- El MISMO problema con PuLP ----------
if PULP_OK:
    # 1) El problema: acá se dice explícitamente que es de MAXIMIZAR (sin trucos de signo)
    modelo = pulp.LpProblem("panaderia", pulp.LpMaximize)

    # 2) Las variables, con su nombre y su cota inferior
    x = pulp.LpVariable("pan",      lowBound=0)
    y = pulp.LpVariable("facturas", lowBound=0)

    # 3) La función objetivo
    modelo += 120*x + 200*y, "ganancia"

    # 4) Las restricciones, con nombre
    modelo += 1*x + 2*y <= 160, "horno"
    modelo += 3*x + 2*y <= 300, "masa"

    # 5) Resolver
    modelo.solve(pulp.PULP_CBC_CMD(msg=0))

    print("Estado:", pulp.LpStatus[modelo.status])
    print(f"Pan:      {x.value():.0f} kg")
    print(f"Facturas: {y.value():.0f} kg")
    print(f"Ganancia: ${pulp.value(modelo.objective):,.0f}   <- positiva, sin dar vuelta el signo")
else:
    print("PuLP no está instalado. Instalalo con: !pip install pulp -q")

## ⚔️ `linprog` contra `PuLP`: pros y contras

Los dos resuelven el mismo problema y dan el mismo resultado. Lo que cambia es **cómo se escribe** y
**qué tan legible queda**:

| | `linprog` (SciPy) | `PuLP` |
|---|---|---|
| **Cómo se plantea** | Con **listas y matrices**: `c`, `A_ub`, `b_ub` | Con **variables con nombre** y ecuaciones |
| **Maximizar** | Hay que poner los coeficientes en **negativo** | Se declara `LpMaximize`, sin trucos |
| **Legibilidad** | Baja: hay que recordar qué columna es cada variable | Alta: se lee casi como el planteo matemático |
| **Ventaja** | Rápido y viene con SciPy, sin instalar nada | Claro, con nombres y restricciones etiquetadas |
| **Desventaja** | Se presta a errores de orden y de signo | Hay que instalarlo (`pip install pulp`) |
| **Variables enteras** | No las maneja | **Sí** (`cat="Integer"`): sirve para "no puedo contratar media persona" |
| **Cuándo usar cada uno** | Problemas chicos, cuentas rápidas | Modelos con varias variables, o si hay que leerlo después |

> 💡 **Si en el parcial piden justificar la elección:** `PuLP` se defiende por **claridad y
> trazabilidad** del modelo (y porque permite variables enteras); `linprog` se defiende por ser
> **directo y sin dependencias**. Las dos respuestas son válidas si se argumenta.

> ⚠️ **Errores clásicos de PL**
>
> | Error | Consecuencia |
> |---|---|
> | Olvidarse del negativo en `c` al maximizar con `linprog` | Se minimiza la ganancia: da la peor solución |
> | Dejar una restricción como $\ge$ cuando `A_ub` espera $\le$ | Hay que multiplicar la fila por $-1$; si no, el modelo es otro |
> | Mezclar unidades (horas con kilos) | El modelo corre igual y el resultado no tiene sentido |
> | No declarar la no negatividad | Aparecen producciones negativas |
> | No interpretar el resultado | Queda un número sin respuesta de gestión: no puntúa |

---
# 🗂️ Cuadros resumen (para los días que quedan)

## Cuadro 1 — Qué herramienta va con cada pregunta

| Si la pregunta es... | La herramienta es | Se ve en la clase |
|---|---|---|
| ¿Cuánto cambia esto si aumenta una unidad? | Derivada: `sp.diff` | 09, 10 |
| ¿Cuál es la cantidad óptima? | Derivar, igualar a cero, verificar 2ª derivada | 11 |
| ¿Cuánto conviene producir con recursos limitados? | Programación lineal: `linprog` / `PuLP` | 08 |
| ¿Cuánto de cada cosa, si todo tiene que cerrar exacto? | Sistema de ecuaciones: `np.linalg.solve` | 07 |
| ¿Cuál es el total de esta tabla, por fila o por columna? | `axis=0` / `axis=1` | 05 |
| ¿Cuánto facturó en total? | Producto matricial `@` | 05 |
| ¿Qué filas cumplen esta condición? | Máscara booleana | 06 |
| ¿Cuánto por categoría? | `groupby` | 06 |
| ¿Dónde se cruzan oferta y demanda? | `sp.solve` sobre la igualdad | 04 |
| ¿Qué me dice este gráfico? | Interpretación: arranque, cruces, máximo, pendiente | 02, 03 |

## Cuadro 2 — Las fórmulas que hay que saber de memoria

| Concepto | Fórmula |
|---|---|
| Costo total | $CT = CF + CV(q)$ |
| Costo medio | $CMe = CT/q$ |
| Costo marginal | $CMg = CT'(q)$ |
| Ingreso total | $IT = p \cdot q$ |
| Ingreso marginal | $IMg = IT'(q)$ |
| Beneficio | $B = IT - CT$ |
| **Óptimo** | $B'(q) = 0$, o bien $IMg = CMg$ |
| Máximo confirmado | $B''(q) < 0$ |
| Mínimo confirmado | $B''(q) > 0$ |
| Punto de nivelación | $IT = CT$, es decir $B = 0$ |
| Elasticidad | $\epsilon_p = \dfrac{\%\Delta q}{\%\Delta p}$ |

## Cuadro 3 — Los pares que se confunden

| | |
|---|---|
| `*` (elemento a elemento) | `@` (multiplica y **suma**: total) |
| `axis=0` (resultado **por columna**) | `axis=1` (resultado **por fila**) |
| `.loc` (por **etiqueta**, final incluido) | `.iloc` (por **posición**, final excluido) |
| `df["col"]` (`Series`) | `df[["col"]]` (`DataFrame`) |
| Costo **medio** (divide) | Costo **marginal** (deriva) |
| `&` (y, entre columnas) | `and` (no funciona con columnas) |
| `linprog` (minimiza siempre) | `PuLP` (se declara `LpMaximize`) |
| 1ª derivada (¿dónde está el candidato?) | 2ª derivada (¿es máximo o mínimo?) |

---
# 🧘 Cómo encarar el parcial

Para muchos de ustedes este es el **primer examen universitario**, y eso pesa más que el contenido.
Van algunas cosas que sirven de verdad.

## Antes de arrancar

- **Leé todo el examen** antes de escribir la primera palabra. Son 4 ejercicios sobre un mismo caso:
  saber para dónde va la historia ayuda a responder el primero.
- **Arrancá por el ejercicio que te sale.** No hay ningún premio por hacerlos en orden, y empezar
  resolviendo algo te ordena la cabeza.
- **Dentro de cada ejercicio, los incisos van de menor a mayor dificultad.** El (a) es el más
  accesible: es un buen lugar para empezar.

## Mientras lo hacés

- **Escribí el planteo, siempre.** Variables con unidades, qué función estás usando y por qué. Toda
  respuesta numérica sin justificación vale **cero**, y el procedimiento puntúa incluso si el número
  final sale mal.
- **Poné las unidades en cada resultado.** "80" no es una respuesta; "80 unidades por día" sí.
- **Cerrá con una frase de interpretación.** Lo que más pesa en la corrección es la lectura
  económica: después de cada cuenta, una línea que diga qué significa para la empresa del caso.
- **Si te trabás, pasá al siguiente.** Quedarse 40 minutos en un inciso es la forma más común de
  perder puntos que estaban a mano. Marcalo y volvé después.
- **Si no te sale la cuenta, explicá el camino.** "Habría que derivar el beneficio, igualarlo a cero
  y verificar con la segunda derivada" muestra que entendiste el procedimiento, y eso se corrige.

## Lo operativo (que también puntúa)

- Son **3 horas**, pensado para resolverse en **2:30**. Los últimos 30 minutos son para **revisar**.
- **Todas las hojas** con nombre, apellido y número de registro, numeradas y ordenadas.
- Indicá claramente **número de ejercicio e inciso** en cada respuesta.
- Se permite **calculadora científica no programable**. No se permiten apuntes, resúmenes, notebooks
  de la cátedra, celular ni asistentes de IA.
- Se admite **redondeo razonable**.

> 💡 **Y algo que no es menor:** respirá. Si te bloqueás, soltá la lapicera treinta segundos y mirá
> otro ejercicio. Le pasa a todo el mundo en el primer parcial, y no dice nada sobre lo que sabés.

---
# 📝 Ejercicios tipo parcial

Cuatro ejercicios con el formato del examen. **Hacelos en papel primero**, sin correr nada, y recién
después abrí la resolución para comparar.

> El caso: **"Molienda del Plata S.A."**, una empresa que produce harina fraccionada. Los cuatro
> ejercicios usan la misma empresa, igual que en el parcial.

### ✅ Ejercicio 1 — Optimización (derivadas)

Molienda del Plata tiene un costo total, en pesos, de $CT(q) = 8000 + 30q + 0{,}25q^2$, donde $q$ son
las toneladas mensuales. Vende cada tonelada según la demanda $p(q) = 250 - 0{,}75q$.

**a)** Escribir las funciones de ingreso total y de beneficio.
**b)** Hallar la cantidad que maximiza el beneficio. **Verificar** que es un máximo.
**c)** ¿Cuál es el beneficio en ese punto? Interpretar el resultado para la empresa.
**d)** El gerente propone producir 100 toneladas "para ganar más volumen". ¿Conviene? Justificar con
el beneficio marginal.

In [ ]:
# Resolvé acá el Ejercicio 1
# Pista: q = sp.symbols("q", positive=True), y después sp.diff / sp.solve

<details>
<summary><b>🔑 Ver resolución del Ejercicio 1</b></summary>

**a)** $IT(q) = p(q)\cdot q = (250 - 0{,}75q)\,q = 250q - 0{,}75q^2$
y $B(q) = IT - CT = 250q - 0{,}75q^2 - 8000 - 30q - 0{,}25q^2 = -q^2 + 220q - 8000$.

**b)** $B'(q) = -2q + 220 = 0 \Rightarrow q = 110$ toneladas.
$B''(q) = -2 < 0$, así que **es un máximo** (condición de segundo orden verificada).

**c)** $B(110) = -(110)^2 + 220(110) - 8000 = -12100 + 24200 - 8000 = \$4.100$ por mes.
Interpretación: el plan de producción que más beneficio deja es **110 toneladas mensuales**, con un
beneficio de \$4.100. Producir más o menos que eso reduce el resultado.

**d)** No conviene quedarse en 100. En $q=100$: $B'(100) = -2(100) + 220 = +20 > 0$, es decir que
la tonelada 101 **todavía suma** \$20 al beneficio. Conviene **subir** hasta 110, no quedarse en 100.
(Y pasadas las 110, $B'$ se vuelve negativo: ahí sí producir más empeora el resultado.)

</details>

### ✅ Ejercicio 2 — Leer una salida de optimización

La empresa puede fraccionar en bolsas de **1 kg** ($x$) y de **5 kg** ($y$), en miles de bolsas por
mes. El área de planeamiento corrió este modelo y trajo la salida:

```python
c = [-40, -150]
A_ub = [[2, 6], [1, 4]]
b_ub = [600, 320]
res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, None), (0, None)], method="highs")
```

```
     message: Optimization terminated successfully.
     success: True
      status: 0
         fun: -12600.0
           x: [ 2.400e+02  2.000e+01]
         nit: 2
```

**a)** ¿Qué está maximizando el modelo y cómo se ve eso en el código?
**b)** ¿Cuánto conviene producir de cada tipo de bolsa?
**c)** ¿Cuál es la ganancia mensual? Explicar el signo de `fun`.
**d)** ¿Las dos restricciones se usan por completo? ¿Qué significa eso para la empresa?

In [ ]:
# Verificá acá tus respuestas del Ejercicio 2

<details>
<summary><b>🔑 Ver resolución del Ejercicio 2</b></summary>

**a)** Maximiza la ganancia $40x + 150y$. Se ve en que los coeficientes de `c` están en
**negativo** (`[-40, -150]`): `linprog` siempre minimiza, así que minimizar $-40x-150y$ es lo mismo
que maximizar $40x+150y$.

**b)** `x: [240, 20]` → **240 mil bolsas de 1 kg** y **20 mil bolsas de 5 kg** por mes. El orden es
el de declaración de las variables: primero $x$ (1 kg), después $y$ (5 kg). El modelo **no** dice
"bolsas de 1 kg": eso lo dice el planteo, y por eso definir las variables con sus unidades es parte
de la respuesta.

**c)** La ganancia es $|{-12600}| = \$12.600$ por mes. `fun` sale **negativo** porque es el valor de
la función que efectivamente se minimizó ($-40x-150y$); no es un error ni una pérdida.

**d)** Hay que reemplazar la solución en cada restricción:

- Restricción 1: $2(240) + 6(20) = 480 + 120 = 600$, contra un límite de **600** → se usa completa.
- Restricción 2: $1(240) + 4(20) = 240 + 80 = 320$, contra un límite de **320** → se usa completa.

Las dos son **restricciones activas**: los dos recursos se agotan. Para la empresa eso significa que
**ambos son cuellos de botella**, y que la única forma de mejorar la ganancia es conseguir más de
alguno de los dos (más horas o más insumo), no reacomodar la mezcla de producción.

> 💡 **Este reemplazo hacelo siempre.** Verificar que la solución cumpla todas las restricciones es
> la forma más rápida de detectar un error de planteo o de lectura de los datos.

</details>

### ✅ Ejercicio 3 — ¿Está bien filtrado?

Un analista quiere quedarse con los registros de **la planta Norte** que superaron las **100
toneladas**, y escribió esto:

```python
filtrado = produccion[produccion["planta"] == "Norte" and produccion["tn"] > 100]
```

**a)** ¿Qué problema tiene esa línea? Corregirla.
**b)** Escribir la versión correcta y decir qué devuelve (una `Series` o un `DataFrame`).
**c)** ¿Qué habría que reportar, además del resultado, para saber si el filtro está bien hecho?

In [ ]:
# Tabla para probar el Ejercicio 3
produccion = pd.DataFrame({
    "planta": ["Norte", "Norte", "Sur", "Norte", "Sur", "Oeste"],
    "mes":    ["ene", "feb", "ene", "mar", "feb", "ene"],
    "tn":     [120, 95, 150, 180, 80, 110],
})

# Escribí acá el filtro corregido

<details>
<summary><b>🔑 Ver resolución del Ejercicio 3</b></summary>

**a)** El problema es el **`and`**: no funciona entre columnas de pandas (tira
`ValueError: The truth value of a Series is ambiguous`). Entre máscaras booleanas va **`&`**, y cada
condición tiene que ir **entre paréntesis**, porque `&` se evalúa antes que `==` y `>`.

**b)** La versión correcta:

```python
filtrado = produccion[(produccion["planta"] == "Norte") & (produccion["tn"] > 100)]
```

Devuelve un **`DataFrame`** (filtrar filas conserva la forma de tabla), con **2 filas**: los meses de
enero (120 tn) y marzo (180 tn) de la planta Norte. Febrero de Norte queda afuera porque 95 no supera
las 100 toneladas.

**c)** Hay que reportar **cuántas filas quedaron sobre el total** (2 de 6) y **qué quedó afuera**: el
febrero de Norte por no llegar a 100 tn, y todas las filas de Sur y Oeste por no ser la planta
pedida. Un filtro sin ese control puede estar dejando afuera datos válidos sin que nadie lo note.

</details>

### ✅ Ejercicio 4 — Integrador

La tabla de producción mensual de la empresa está cargada en una matriz `P`, donde las **filas son
las tres plantas** (Norte, Sur, Oeste) y las **columnas los cuatro meses**:

```python
P = np.array([[120,  95, 180, 140],
              [150,  80, 130, 160],
              [110, 105,  90, 120]])
precios = np.array([2000, 2100, 2050, 2200])   # precio por tonelada, cada mes
```

**a)** ¿Qué devuelve `P.sum(axis=1)` y qué pregunta de gestión responde?
**b)** ¿Y `P.sum(axis=0)`?
**c)** Escribir la operación que da la **facturación total** de la empresa en los cuatro meses, y
explicar por qué va `@` y no `*`.
**d)** La gerencia quiere saber qué planta facturó más. ¿Alcanza con mirar la producción total?
Justificar.

In [ ]:
# Probá acá el Ejercicio 4
P = np.array([[120,  95, 180, 140],
              [150,  80, 130, 160],
              [110, 105,  90, 120]])
precios = np.array([2000, 2100, 2050, 2200])

<details>
<summary><b>🔑 Ver resolución del Ejercicio 4</b></summary>

**a)** `P.sum(axis=1)` = `[535, 520, 425]`: **un valor por planta** (se consume el eje de las
columnas, o sea los meses). Responde: *¿cuánto produjo cada planta en todo el período?*

**b)** `P.sum(axis=0)` = `[380, 280, 400, 420]`: **un valor por mes** (se consume el eje de las
filas, las plantas). Responde: *¿cuánto produjo la empresa cada mes?*

**c)** La facturación total es `P.sum(axis=0) @ precios` = **\$3.092.000**: primero el total
producido **por mes**, y después cada mes se multiplica por **su** precio y se suma todo. Va `@`
porque el resultado tiene que ser **un total** (multiplica y suma). Con `*` quedarían cuatro números
—la facturación de cada mes— y no el total. *(También sirve `(P * precios).sum()`, que multiplica
cada celda por el precio de su mes y suma todo: da el mismo \$3.092.000.)*

**d)** No alcanza. La producción total ignora que **el precio cambia mes a mes**: una planta que
produjo más en los meses baratos puede facturar menos que otra que produjo menos pero en los meses
caros. Hay que calcular `P @ precios`, que da la facturación **por planta** teniendo en cuenta el
precio de cada mes: `[1.116.500, 1.086.500, 889.000]`. En este caso el orden coincide con el de la
producción (Norte, Sur, Oeste), pero eso hay que **verificarlo**, no suponerlo: la diferencia entre
Norte y Sur se acorta bastante (535 contra 520 toneladas, pero \$1.116.500 contra \$1.086.500).

</details>

---
# 🧭 Para llevarse

1. **El parcial no pide escribir código, pide leerlo e interpretarlo.** Frente a un bloque, la
   pregunta siempre es la misma: *¿qué hace, qué devuelve y qué significa para la organización?*
2. **Ningún número viaja solo.** Unidades y una frase de interpretación: es la parte que más pesa en
   la corrección, y la que más se olvida.
3. **La receta de optimización es siempre la misma:** plantear → derivar e igualar a cero → verificar
   con la segunda derivada → interpretar. El paso 3 no se saltea.
4. **Los pares que se confunden** (`*`/`@`, `axis`, `.loc`/`.iloc`, medio/marginal, `linprog`/`PuLP`)
   son material de examen justamente porque se confunden. El Cuadro 3 es para eso.
5. **Verificá siempre.** Que la solución cumpla las restricciones, que los grupos sumen el total, que
   el filtro deje la cantidad de filas que esperabas. La mitad de los errores se cazan reemplazando.

### 📚 Para seguir repasando

- Las notebooks de las clases **00 a 11**, que son la fuente de todo lo que está acá.
- Los **ejercicios integradores** de las clases 03, 05, 07 y 11.
- Las **pautas del parcial** publicadas en el campus: fecha, materiales permitidos y criterios de
  corrección.

**¡Éxitos el martes!** 🍀